# Toy manifolds Uniform Generation Figures

Renders the four single-seed panels per manifold from the volume-guided
generation runs:

1. biased training density in the chart
2. density after volume-guided Langevin
3. distance to the manifold across the chart
4. KDE density vs analytic volume element

Runs are read from
`results/toy_manifolds_test/uniform_generation/{manifold}/seed_{seed}/`
(set `RESULTS_ROOT_NAME` below; switch it to `toy_manifolds` for the paper
runs). This notebook only renders: every per-point quantity (KDE colors,
distances, volume element, density, plot mask) is either read from
`figure_data.npz` or rebuilt from `generated_points.pt` when the run was
written with `save_figure_data: false`.


## Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import gaussian_filter

%matplotlib inline


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "experiments" / "toy_manifolds").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root (expected experiments/toy_manifolds/)."
    )


REPO_ROOT = _find_repo_root()

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from experiments.toy_manifolds.common import (
    MANIFOLDS,
    evaluate_uniformity,
    fit_kde,
    manifold_distance,
    sample_imbalanced,
)
from geometry_distill.seeding import set_seed

RESULTS_ROOT_NAME = "toy_manifolds"  # "toy_manifolds" for the paper runs
RESULTS_ROOT = REPO_ROOT / "results" / RESULTS_ROOT_NAME
FIG_ROOT = REPO_ROOT / "notebooks" / "figures" / "toy_uniform_generation"

MANIFOLD_NAMES = ("hemisphere", "saddle", "paraboloid")
SEED = 43  # seed shown in the figure panels


def run_dir(name: str, seed: int) -> Path:
    return RESULTS_ROOT / "uniform_generation" / name / f"seed_{seed}"


def available_seeds(name: str) -> list[int]:
    return sorted(
        int(path.name.removeprefix("seed_"))
        for path in (RESULTS_ROOT / "uniform_generation" / name).glob("seed_*")
        if path.is_dir()
    )


for _name in MANIFOLD_NAMES:
    print(f"{_name:<12} {run_dir(_name, SEED).relative_to(REPO_ROOT)}")


## Rendering helpers

In [ ]:
def _rel(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def build_figure_data(
    name: str, points: torch.Tensor, config: dict, evaluation: dict
) -> dict:
    """Rebuild the per-point arrays of ``figure_data.npz`` from saved points."""
    manifold = MANIFOLDS[name]
    set_seed(int(config["seed"]))
    _, uv_biased = sample_imbalanced(manifold, n_points=int(config["figure_bias_n"]))
    uv_gen = manifold.inverse(points[:, 0], points[:, 1], points[:, 2])

    c_biased = np.asarray(fit_kde(uv_biased)(uv_biased), dtype=np.float64)
    c_gen = np.asarray(fit_kde(uv_gen)(uv_gen), dtype=np.float64)

    uv_gen_np = uv_gen.detach().cpu().numpy()
    points_biased = manifold.parametrization(uv_biased[:, 0], uv_biased[:, 1])
    R_biased = float(evaluate_uniformity(points_biased, manifold)["R"])

    eval_arrays = evaluate_uniformity(points, manifold, return_arrays=True)
    domain = str(config["figure_plot_domain"])
    half_width = float(config["figure_plot_half_width"])
    if domain == "disk":
        plot_mask = (
            uv_gen_np[:, 0] ** 2 + uv_gen_np[:, 1] ** 2
        ) <= half_width**2 + 1e-6
    else:
        plot_mask = (np.abs(uv_gen_np[:, 0]) <= half_width + 1e-6) & (
            np.abs(uv_gen_np[:, 1]) <= half_width + 1e-6
        )

    return {
        "uv_biased": uv_biased.detach().cpu().numpy(),
        "c_biased": c_biased,
        "uv_gen": uv_gen_np,
        "c_gen": c_gen,
        "dists_all": np.asarray(manifold_distance(points, manifold), dtype=np.float64),
        "plot_mask": plot_mask,
        "plot_half_width": np.asarray(half_width),
        "plot_domain": np.asarray(domain),
        "R_biased": np.asarray(R_biased, dtype=np.float64),
        "R_langevin": np.asarray(float(evaluation["R"]), dtype=np.float64),
        "uv_masked": eval_arrays["uv_masked"].detach().cpu().numpy(),
        "density": np.asarray(eval_arrays["density"], dtype=np.float64),
        "vol_elem": np.asarray(eval_arrays["vol_elem"], dtype=np.float64),
    }


def load_run(name: str, seed: int) -> dict:
    """Load one run's figure arrays, evaluation and config."""
    run = run_dir(name, seed)
    config_path = run / "config.json"
    eval_path = run / "evaluation.json"
    points_path = run / "generated_points.pt"
    for path in (config_path, eval_path, points_path):
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing artifact for '{name}' seed {seed}: {path}. Run "
                "`python -m experiments.toy_manifolds.scripts.reproduce "
                "--config experiments/toy_manifolds/configs/reproduce_test.json` first."
            )
    config = json.loads(config_path.read_text())
    evaluation = json.loads(eval_path.read_text())

    fig_path = run / "figure_data.npz"
    if fig_path.is_file():
        with np.load(fig_path) as z:
            data = {key: z[key] for key in z.files}
    else:
        points = torch.load(points_path, map_location="cpu", weights_only=False)[
            "points_3d"
        ]
        data = build_figure_data(name, points, config, evaluation)

    data["evaluation"] = evaluation
    data["config"] = config
    data["run_dir"] = run
    return data


def masked_distance_mean(name: str, seed: int) -> float:
    """Mean true distance to the manifold over points inside the evaluation mask."""
    points = torch.load(
        run_dir(name, seed) / "generated_points.pt",
        map_location="cpu",
        weights_only=False,
    )["points_3d"]
    manifold = MANIFOLDS[name]
    uv = manifold.inverse(points[:, 0], points[:, 1], points[:, 2])
    mask = manifold.boundary_mask(uv[:, 0], uv[:, 1])
    return float(manifold_distance(points[mask], manifold).mean())


def save_fig(fig: plt.Figure, stem: str, out_dir: Path) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    png = out_dir / f"{stem}.png"
    svg = out_dir / f"{stem}.svg"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    print("wrote", _rel(png))
    print("wrote", _rel(svg))
    return [png, svg]


def _annotate_R(ax, R: float, domain: str, half_width: float) -> None:
    """R label in the top-left of the plot domain."""
    if domain == "disk":
        x, y = -0.92, 0.92
    else:
        x, y = -(half_width - 0.05), half_width - 0.05
    ax.text(
        x,
        y,
        rf"$R={R:+.2f}$",
        transform=ax.transData,
        ha="left",
        va="top",
        fontsize=10,
        color="black",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 2.0},
        zorder=5,
    )


def smooth_scalar_field(
    uv: np.ndarray,
    values: np.ndarray,
    domain: str,
    half_width: float,
    n_grid: int = 320,
    bandwidth: float | None = None,
):
    """Nadaraya-Watson kernel smoother -> continuous field on the plot domain."""
    ax_pad = half_width * 1.08
    eps = 1e-6
    u_g = np.linspace(-ax_pad, ax_pad, n_grid)
    v_g = np.linspace(-ax_pad, ax_pad, n_grid)
    U, V = np.meshgrid(u_g, v_g, indexing="xy")
    grid = np.stack([U.ravel(), V.ravel()], axis=-1)
    if domain == "disk":
        on_domain = (grid[:, 0] ** 2 + grid[:, 1] ** 2) <= half_width**2 + eps
    else:
        on_domain = (np.abs(grid[:, 0]) <= half_width + eps) & (
            np.abs(grid[:, 1]) <= half_width + eps
        )

    if bandwidth is None:
        # Scott-like scale from the point cloud
        bandwidth = float(np.std(uv, axis=0).mean() * uv.shape[0] ** (-1.0 / 6.0))
        bandwidth = max(bandwidth, 0.05)

    field = np.full(grid.shape[0], np.nan, dtype=np.float64)
    idx = np.where(on_domain)[0]
    chunk = 2048
    inv_2h2 = 1.0 / (2.0 * bandwidth**2)
    for start in range(0, len(idx), chunk):
        sel = idx[start : start + chunk]
        d2 = ((grid[sel, None, :] - uv[None, :, :]) ** 2).sum(axis=-1)
        w = np.exp(-d2 * inv_2h2)
        w_sum = w.sum(axis=1)
        field[sel] = np.where(w_sum > 0, (w * values[None, :]).sum(axis=1) / w_sum, np.nan)
    return U, V, field.reshape(U.shape)


## Panel renderer

In [ ]:
def render_manifold(name: str) -> dict:
    """Render and save the four single-seed panels for one manifold run."""
    data = load_run(name, SEED)
    ev = data["evaluation"]

    uv_biased = np.asarray(data["uv_biased"])
    c_biased = np.asarray(data["c_biased"])
    uv_gen = np.asarray(data["uv_gen"])
    c_gen = np.asarray(data["c_gen"])
    dists_all = np.asarray(data["dists_all"])
    keep = np.asarray(data["plot_mask"], dtype=bool)

    domain = str(data["plot_domain"])
    half_width = float(data["plot_half_width"])
    ax_pad = half_width * 1.08

    R_biased = float(data["R_biased"])
    R_langevin = float(data["R_langevin"])
    R2 = float(ev["R2"])

    run_id = f"seed_{SEED}"
    out_dir = FIG_ROOT / name / run_id
    saved: list[Path] = []

    # 1) Biased start -- all points, coloured by local KDE
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    sc = ax.scatter(
        uv_biased[:, 0], uv_biased[:, 1], c=c_biased, s=4, cmap="magma",
        alpha=0.75, edgecolors="none", rasterized=True,
    )
    ax.set_aspect("equal")
    ax.set_xlim(-ax_pad, ax_pad)
    ax.set_ylim(-ax_pad, ax_pad)
    ax.set_xlabel(r"$u$")
    ax.set_ylabel(r"$v$")
    ax.set_title("Biased training density (chart)")
    _annotate_R(ax, R_biased, domain, half_width)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="local density")
    fig.tight_layout()
    saved += save_fig(fig, f"{name}_biased_density", out_dir)

    # 2) After Langevin -- points inside the plot domain
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    sc = ax.scatter(
        uv_gen[keep, 0], uv_gen[keep, 1], c=c_gen[keep], s=4, cmap="magma",
        alpha=0.75, edgecolors="none", rasterized=True,
    )
    ax.set_aspect("equal")
    ax.set_xlim(-ax_pad, ax_pad)
    ax.set_ylim(-ax_pad, ax_pad)
    ax.set_xlabel(r"$u$")
    ax.set_ylabel(r"$v$")
    ax.set_title("Density after volume-guided Langevin")
    _annotate_R(ax, R_langevin, domain, half_width)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="local density")
    fig.tight_layout()
    saved += save_fig(fig, f"{name}_langevin_density", out_dir)

    # 3) Distance -- smoothed scalar field on the plot domain
    U_d, V_d, dist_field = smooth_scalar_field(
        uv_gen[keep], dists_all[keep], domain, half_width
    )
    domain_mask = np.isfinite(dist_field)
    fill = np.where(domain_mask, dist_field, 0.0)
    blurred = gaussian_filter(fill, sigma=1.2)
    blurred = np.where(domain_mask, blurred, np.nan)

    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    levels = np.linspace(np.nanmin(blurred), np.nanmax(blurred), 64)
    mesh = ax.contourf(U_d, V_d, blurred, levels=levels, cmap="magma", antialiased=True)
    ax.set_aspect("equal")
    ax.set_xlim(-ax_pad, ax_pad)
    ax.set_ylim(-ax_pad, ax_pad)
    ax.set_xlabel(r"$u$")
    ax.set_ylabel(r"$v$")
    if domain == "disk":
        ax.set_title("Distance to manifold across the disk")
        stem = f"{name}_distance_across_disk"
    else:
        ax.set_title("Distance to manifold across the chart")
        stem = f"{name}_distance_across_chart"
    fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="distance")
    fig.tight_layout()
    saved += save_fig(fig, stem, out_dir)

    # 4) KDE vs analytic volume element
    vol = np.asarray(data["vol_elem"], dtype=np.float64)
    dens = np.asarray(data["density"], dtype=np.float64)
    slope, intercept = np.polyfit(vol, dens, 1)
    xs = np.linspace(vol.min(), vol.max(), 100)

    fig, ax = plt.subplots(figsize=(4.4, 3.8))
    ax.scatter(vol, dens, s=8, alpha=0.45, c="#4c72b0", edgecolors="none")
    ax.plot(xs, intercept + slope * xs, color="#c0392b", lw=1.6, label="linear fit")
    ax.set_xlabel(r"analytic volume element $f_{\mathrm{vol}}$")
    ax.set_ylabel("KDE density of Langevin samples")
    ax.set_title("KDE vs analytic volume element")
    ax.text(
        0.04,
        0.96,
        rf"$R={R_langevin:+.3f}$" + "\n" + rf"$R^{2}={R2:.3f}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 2.0},
    )
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    saved += save_fig(fig, f"{name}_kde_vs_volume", out_dir)

    dist_masked = masked_distance_mean(name, SEED)
    print(
        f"[{name}] R={R_langevin:+.4f}  R2={R2:.4f}  "
        f"dist_masked={dist_masked:.4f}  n_masked={ev['n_masked']}"
    )
    return {
        "manifold": name,
        "seed": SEED,
        "R": R_langevin,
        "R2": R2,
        "dist_masked_mean": dist_masked,
        "n_masked": int(ev["n_masked"]),
        "figures": [_rel(p) for p in saved],
    }


### Hemisphere

In [ ]:
render_manifold("hemisphere")

### Saddle

In [ ]:
render_manifold("saddle")

### Paraboloid

In [ ]:
render_manifold("paraboloid")

## Summary


In [ ]:
rows = []
manifest: dict = {
    "results_root": _rel(RESULTS_ROOT),
    "figures_root": _rel(FIG_ROOT),
    "manifolds": {},
}

for name in MANIFOLD_NAMES:
    gen_eval_path = run_dir(name, SEED) / "evaluation.json"
    if not gen_eval_path.is_file():
        print(f"skip {name}: no evaluation at {_rel(gen_eval_path)}")
        continue
    evaluation = json.loads(gen_eval_path.read_text())
    proj_eval_path = (
        RESULTS_ROOT / "projector" / name / f"seed_{SEED}" / "evaluation.json"
    )
    proj_mse = None
    if proj_eval_path.is_file():
        proj_mse = json.loads(proj_eval_path.read_text()).get("proj_mse")
    fig_dir = FIG_ROOT / name / f"seed_{SEED}"
    figures = sorted(
        _rel(p) for p in fig_dir.glob("*") if p.suffix in (".png", ".svg")
    )
    dist_masked = masked_distance_mean(name, SEED)
    manifest["manifolds"][name] = {
        "seed": SEED,
        "evaluation": evaluation,
        "dist_masked_mean": dist_masked,
        "figures": figures,
    }
    rows.append(
        {
            "manifold": name,
            "seed": SEED,
            "R": evaluation["R"],
            "R2": evaluation["R2"],
            "dist_masked_mean": dist_masked,
            "n_masked": evaluation["n_masked"],
            "proj_mse": proj_mse,
        }
    )

if rows:
    display(pd.DataFrame(rows))

_manifest_path = FIG_ROOT / "figures_manifest.json"
_manifest_path.parent.mkdir(parents=True, exist_ok=True)
_manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("wrote", _rel(_manifest_path))


## Summary table across seeds


In [ ]:
from IPython.display import HTML, display

GAGA = {
    "hemisphere": {"R": 0.85, "R2": 0.71},
    "saddle": {"R": 0.60, "R2": 0.36},
    "paraboloid": {"R": 0.66, "R2": 0.44},
}
MANIFOLD_LABELS = {
    "hemisphere": "Hemisphere",
    "saddle": "Saddle",
    "paraboloid": "Paraboloid",
}
METRICS = ("R", "R2")
ROW_LABELS = ("Biased", "Ours", "GAGA")

# None uses the score model's n_train; 5000 matches the figure protocol.
BIASED_N: int | None = None


def _score_n_train(name: str, seed: int) -> int:
    config_path = RESULTS_ROOT / "score_model" / name / f"seed_{seed}" / "config.json"
    return int(json.loads(config_path.read_text())["n_train"])


biased_per_seed: dict[str, dict[str, list[float]]] = {}
for name in MANIFOLD_NAMES:
    seeds = available_seeds(name)
    if not seeds:
        print(
            f"skip {name}: no runs under {_rel(RESULTS_ROOT / 'uniform_generation' / name)}"
        )
        continue
    n_biased = BIASED_N or _score_n_train(name, seeds[0])
    for seed in seeds:
        set_seed(seed)
        points_biased, _ = sample_imbalanced(MANIFOLDS[name], n_points=n_biased)
        res = evaluate_uniformity(points_biased, MANIFOLDS[name])
        biased_per_seed.setdefault(name, {"R": [], "R2": []})
        biased_per_seed[name]["R"].append(float(res["R"]))
        biased_per_seed[name]["R2"].append(float(res["R2"]))

per_seed: dict[str, dict[str, list[float]]] = {}
for name in MANIFOLD_NAMES:
    for seed in available_seeds(name):
        eval_path = run_dir(name, seed) / "evaluation.json"
        if not eval_path.is_file():
            print(f"skip {name} seed {seed}: missing {_rel(eval_path)}")
            continue
        ev = json.loads(eval_path.read_text())
        per_seed.setdefault(name, {"R": [], "R2": [], "dist_masked": []})
        per_seed[name]["R"].append(float(ev["R"]))
        per_seed[name]["R2"].append(float(ev["R2"]))
        per_seed[name]["dist_masked"].append(masked_distance_mean(name, seed))

table: list[dict] = []
for name in MANIFOLD_NAMES:
    values = per_seed.get(name)
    if not values:
        raise RuntimeError(f"No seed evaluations found for '{name}'.")
    rows = {
        "Biased": {m: float(np.mean(biased_per_seed[name][m])) for m in METRICS},
        "Ours": {m: float(np.mean(values[m])) for m in METRICS},
        "GAGA": {m: GAGA[name][m] for m in METRICS},
    }
    std = {
        "Biased": {m: float(np.std(biased_per_seed[name][m], ddof=0)) for m in METRICS},
        "Ours": {m: float(np.std(values[m], ddof=0)) for m in METRICS},
    }
    best = {m: max(rows[label][m] for label in ROW_LABELS) for m in METRICS}
    dist = {
        # Biased samples are drawn on the analytic manifold, so this is exact.
        "Biased": (0.0, None),
        "Ours": (
            float(np.mean(values["dist_masked"])),
            float(np.std(values["dist_masked"], ddof=0)),
        ),
        "GAGA": None,
    }
    table.append(
        {
            "manifold": name,
            "label": MANIFOLD_LABELS.get(name, name.title()),
            "n_seeds": len(values["R"]),
            "rows": rows,
            "std": std,
            "best": best,
            "dist": dist,
        }
    )


def _html_cell(entry: dict, label: str, metric: str) -> str:
    if label == "GAGA":
        text = f"{entry['rows']['GAGA'][metric]:.2f}"
    else:
        text = f"{entry['rows'][label][metric]:.3f} ± {entry['std'][label][metric]:.3f}"
    if np.isclose(entry["rows"][label][metric], entry["best"][metric]):
        text = f"<b>{text}</b>"
    return text


def _html_dist(entry: dict, label: str) -> str:
    value = entry["dist"][label]
    if value is None:
        return "—"
    mean, std = value
    if std is None:
        return f"{mean:.3f}"
    return f"{mean:.3f} ± {std:.3f}"


html = [
    "<table>",
    (
        "<thead><tr><th>Manifold</th><th>Data</th><th>R</th><th>R²</th>"
        "<th>Distance to true manifold</th></tr></thead>"
    ),
    "<tbody>",
]
n_rows = len(ROW_LABELS)
for i, entry in enumerate(table):
    for j, label in enumerate(ROW_LABELS):
        first = f'<td rowspan="{n_rows}"><b>{entry["label"]}</b></td>' if j == 0 else ""
        html.append(
            f"<tr>{first}<td>{label}</td>"
            f"<td>{_html_cell(entry, label, 'R')}</td>"
            f"<td>{_html_cell(entry, label, 'R2')}</td>"
            f"<td>{_html_dist(entry, label)}</td></tr>"
        )
    if i < len(table) - 1:
        html.append('<tr><td colspan="5" style="border-top:1px solid #888"></td></tr>')
html += ["</tbody>", "</table>"]
display(HTML("\n".join(html)))


LATEX_ROW_LABELS = {
    "Biased": "Training data",
    "GAGA": r"\gaga",
    "Ours": "Ours",
}
LATEX_BLOCKS = (
    ("R", r"\textit{Pearson $R$}"),
    ("R2", r"\textit{$R^2$}"),
)


def _latex_correlation(entry: dict, label: str, metric: str) -> str:
    inner = f"{entry['rows'][label][metric]:.2f}"
    if label == "Ours":
        inner += f" \\pm {entry['std'][label][metric]:.2f}"
    if np.isclose(entry["rows"][label][metric], entry["best"][metric]):
        inner = rf"\mathbf{{{inner}}}"
    return rf"\textcolor{{red}}{{${inner}$}}"


def _latex_distance(entry: dict, label: str) -> str:
    value = entry["dist"][label]
    if value is None:
        return ""
    mean, std = value
    if std is None:
        inner = f"{mean:.4f}"
    else:
        inner = f"{mean:.4f} \\pm {std:.4f}"
    return rf"\textcolor{{red}}{{${inner}$}}"


n_seeds = table[0]["n_seeds"]
lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\small",
    r"\setlength{\tabcolsep}{4pt}",
    r"\begin{tabular}{lccc}",
    r"\toprule",
    " & " + " & ".join(entry["label"] for entry in table) + r" \\",
    r"\midrule",
]
for metric, heading in LATEX_BLOCKS:
    lines.append(rf"\multicolumn{{4}}{{l}}{{{heading}}} \\")
    for label in ("Biased", "GAGA", "Ours"):
        cells = " & ".join(_latex_correlation(entry, label, metric) for entry in table)
        lines.append(rf"{LATEX_ROW_LABELS[label]:<14} & {cells} \\")
    lines.append(r"\midrule")
lines.append(r"\multicolumn{4}{l}{\textit{Distance to true manifold}} \\")
for label in ("Biased", "GAGA", "Ours"):
    cells = " & ".join(_latex_distance(entry, label) for entry in table)
    lines.append(rf"{LATEX_ROW_LABELS[label]:<14} & {cells} \\")
lines += [
    r"\bottomrule",
    r"\end{tabular}",
    (
        r"\caption{Correlation between the density of the samples and the "
        r"analytic volume element. Displayed are mean $\pm$ standard deviation "
        rf"values over {n_seeds} seeds; Values for the \gaga method are taken "
        r"from their original publication.}"
    ),
    r"\label{tab:toy_manifolds}",
    r"\end{table}",
]
latex_table = "\n".join(lines)
print(latex_table)
